# 01 · Data Preparation

실행 방식 1: 노트북 only.

[MovieLens 25M](https://files.grouplens.org/datasets/movielens/ml-25m.zip) (~250MB)을 다운로드해 implicit feedback 학습 데이터로 변환합니다. 결과는 `DATA_DIR/{train,val}/shard-*.parquet`와 `DATA_DIR/_meta.json`에 저장됩니다.

| 단계 | 처리 |
|------|------|
| download | `/local_disk0`에 zip → `ml-25m/ratings.csv` 추출 |
| 라벨 변환 | `rating >= 4` → positive (label=1) |
| negative sampling | positive 당 1개, uniform 랜덤 movie (1:1 ratio) |
| index remap | userId, movieId → 0-based contiguous index |
| split | 행 단위 random 10% → val |
| 저장 | UC Volume에 shard 단위 parquet, `_meta.json`에 `n_users`/`n_items`/카운트 기록 |

자세한 설명은 [`00-foundations/data-loading.md`](../00-foundations/data-loading.md). 모든 학습 노트북(`02..05`)이 같은 `DATA_DIR` 경로를 읽습니다.

In [ ]:
%run ./00-setup

## 다운로드 + 압축 해제

`/local_disk0`은 driver 노드의 로컬 SSD입니다. UC Volume에 바로 풀면 unzip I/O가 느립니다. 노드 로컬에 unzip 후 `ratings.csv`만 사용.

In [ ]:
import os
import urllib.request
import zipfile

EXTRACT_ROOT = "/local_disk0/ml-25m-extract"
ZIP_PATH = "/local_disk0/ml-25m.zip"
RATINGS_CSV = os.path.join(EXTRACT_ROOT, "ml-25m", "ratings.csv")

if not os.path.exists(RATINGS_CSV):
    if not os.path.exists(ZIP_PATH):
        print(f"downloading {ML25M_URL} → {ZIP_PATH}")
        urllib.request.urlretrieve(ML25M_URL, ZIP_PATH)
    print(f"extracting {ZIP_PATH} → {EXTRACT_ROOT}")
    os.makedirs(EXTRACT_ROOT, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_ROOT)
else:
    print(f"reuse {RATINGS_CSV}")

print(f"ratings.csv size: {os.path.getsize(RATINGS_CSV) / 1e6:.1f} MB")

## 라벨 변환 + index remap + negative sampling

`ratings.csv`를 pandas로 한 번에 로드합니다 (~600MB). 그 위에서 작업:

1. **dense remap**: `userId`, `movieId`를 0-based contiguous int64 인덱스로 변환. embedding lookup 효율 + vocab 크기 명시화.
2. **positive**: `rating >= 4`인 (user, item) 쌍을 label=1.
3. **negative sampling**: 같은 user에 대해 uniform 랜덤 movieIdx 1개를 label=0으로 추가. ML-25M density(~0.26%)가 매우 sparse해서 negative가 우연히 positive와 충돌할 확률은 무시 가능.
4. **shuffle**: train 분포가 한쪽으로 쏠리지 않도록 전체 셔플.

In [ ]:
import numpy as np
import pandas as pd

SEED = 0

ratings = pd.read_csv(RATINGS_CSV)
print(f"ratings.csv loaded: {len(ratings):,} rows, columns={list(ratings.columns)}")

unique_users = np.sort(ratings["userId"].unique())
unique_items = np.sort(ratings["movieId"].unique())
user_to_idx = pd.Series(np.arange(len(unique_users), dtype=np.int64), index=unique_users)
item_to_idx = pd.Series(np.arange(len(unique_items), dtype=np.int64), index=unique_items)
n_users = len(unique_users)
n_items = len(unique_items)
print(f"n_users={n_users:,} n_items={n_items:,}")

pos_mask = ratings["rating"] >= 4.0
pos = pd.DataFrame({
    "user_id": user_to_idx.loc[ratings.loc[pos_mask, "userId"]].to_numpy(),
    "item_id": item_to_idx.loc[ratings.loc[pos_mask, "movieId"]].to_numpy(),
    "label": np.ones(int(pos_mask.sum()), dtype=np.float32),
})
print(f"positives: {len(pos):,} ({len(pos)/len(ratings):.1%} of ratings)")

rng = np.random.default_rng(SEED)
neg = pd.DataFrame({
    "user_id": pos["user_id"].to_numpy(),
    "item_id": rng.integers(0, n_items, size=len(pos), dtype=np.int64),
    "label": np.zeros(len(pos), dtype=np.float32),
})

interactions = pd.concat([pos, neg], ignore_index=True)
interactions = interactions.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print(f"interactions (pos+neg): {len(interactions):,}")

# 원본 ratings 메모리 해제 (driver RAM 절약)
del ratings, pos, neg

## shard 단위 parquet write + metadata

driver RAM에 한 번에 다 있긴 하지만, 학습 노트북이 train/val을 lazy하게 묶어 읽을 수 있도록 shard로 잘라 UC Volume에 저장합니다. shard별로 행마다 `val_ratio` 확률로 val로 분리.

마지막에 `n_users`, `n_items`, 행 수를 `_meta.json`에 기록 → 00-setup이 학습 시 자동 로드.

In [ ]:
import json
import shutil

import pyarrow as pa
import pyarrow.parquet as pq

VAL_RATIO = CONFIG["val_ratio"]
ROWS_PER_SHARD = 1_000_000

# 기존 출력 디렉토리 초기화
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
train_dir = os.path.join(DATA_DIR, "train")
val_dir = os.path.join(DATA_DIR, "val")
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

split_rng = np.random.default_rng(SEED + 1)
n_train_rows = 0
n_val_rows = 0
total = len(interactions)
n_shards = (total + ROWS_PER_SHARD - 1) // ROWS_PER_SHARD

for shard_idx in range(n_shards):
    start = shard_idx * ROWS_PER_SHARD
    end = min(start + ROWS_PER_SHARD, total)
    chunk = interactions.iloc[start:end]
    is_val = split_rng.random(size=len(chunk)) < VAL_RATIO
    for split, mask, target_dir in [
        ("train", ~is_val, train_dir),
        ("val", is_val, val_dir),
    ]:
        if not mask.any():
            continue
        sub = chunk.loc[mask]
        table = pa.table({
            "user_id": sub["user_id"].to_numpy(),
            "item_id": sub["item_id"].to_numpy(),
            "label": sub["label"].to_numpy(),
        })
        pq.write_table(table, os.path.join(target_dir, f"shard-{shard_idx:05d}.parquet"))
        if split == "train":
            n_train_rows += int(mask.sum())
        else:
            n_val_rows += int(mask.sum())

meta = {
    "dataset": "movielens-25m",
    "n_users": int(n_users),
    "n_items": int(n_items),
    "n_train_rows": int(n_train_rows),
    "n_val_rows": int(n_val_rows),
    "val_ratio": VAL_RATIO,
    "neg_ratio": 1.0,
    "label_threshold": 4.0,
    "seed": SEED,
}
with open(os.path.join(DATA_DIR, "_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
print(f"wrote {n_shards} shard pair(s)")
print(f"train={n_train_rows:,} val={n_val_rows:,}")
print(f"meta: {meta}")

## 확인

생성된 train shard 한 개를 로드해 스키마와 분포를 확인합니다. label mean은 negative sampling 비율이 1:1이므로 0.5 근처여야 합니다.

In [ ]:
train_shards = sorted(f for f in os.listdir(train_dir) if f.endswith(".parquet"))
val_shards = sorted(f for f in os.listdir(val_dir) if f.endswith(".parquet"))
print(f"train shards: {len(train_shards)}")
print(f"val   shards: {len(val_shards)}")

sample = pq.read_table(os.path.join(train_dir, train_shards[0])).to_pandas()
print(sample.head())
print(f"train shard[0] label mean = {sample['label'].mean():.3f}")
print(f"train shard[0] user_id range = [{sample['user_id'].min()}, {sample['user_id'].max()}]")
print(f"train shard[0] item_id range = [{sample['item_id'].min()}, {sample['item_id'].max()}]")